# Non-Linear Preprocessing Comparison

In [7]:
import os
import pandas as pd

In [8]:
note_path = os.getcwd()
repo_path = os.path.abspath(os.path.join(note_path, ".."))
data_path = os.path.join(repo_path, "data")

In [16]:
path = os.path.join(data_path, "tmp.parquet")
df_tmp = (pd
    .read_parquet(path = path, engine = "pyarrow")
    [["SPY"]]
    .assign(
        spy_rtn    = lambda x: x.SPY.pct_change(),
        px_diff    = lambda x: x.SPY.diff(),
        fast_trend = lambda x: x.px_diff.ewm(span = 10, adjust = False).mean(),
        med_trend  = lambda x: x.px_diff.ewm(span = 30, adjust = False).mean(),
        slow_trend = lambda x: x.px_diff.ewm(span = 100, adjust = False).mean())
    .drop(columns = ["px_diff", "SPY"]))

In [21]:
df_input = (df_tmp
    .reset_index()
    .set_index(["date", "spy_rtn"])
    .shift()
    .reset_index()
    .set_index("date")
    .dropna())

In [81]:
df_input.columns.name = None

In [82]:
target   = "spy_rtn"
features = [col for col in df_input.columns if col != "spy_rtn"] 

In [83]:
from sklearn.feature_selection import mutual_info_regression

In [84]:
mi = (mutual_info_regression(
    X            = df_input[features],
    y            = df_input[[target]],
    random_state = 42))

C:\Users\Diego\anaconda3\Lib\site-packages\sklearn\utils\validation.py:1406: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


In [85]:
df_mi = (pd
    .DataFrame(
        data    = mi,
        index   = features,
        columns = ["mi"])
    .reset_index()
    .rename(columns = {"index": "features"}))

In [88]:
df_setup = (df_input
    .reset_index()
    .melt(id_vars = ["date", target], var_name = "features")
    .drop(columns = ["date"]))

df_pearson = (df_setup
    .groupby("features")
    .agg("corr")
    .loc[lambda x: x.value != 1]
    [["value"]]
    .reset_index()
    [["features", "value"]]
    .rename(columns = {
        "value"   : "pearson",
        "features": "features"}))

In [89]:
df_spearman = (df_setup
    .groupby("features")
    .agg("corr", method = "spearman")
    .loc[lambda x: x.value != 1]
    .reset_index()
    [["features", "value"]]
    .rename(columns = {
        "value" : "spearman",
        "features": "features"}))

In [75]:
import dcor

In [91]:
df_dcor = pd.DataFrame({
    "features": features,
    "dcor": [
        dcor.distance_correlation(
            df_input[feature],
            df_input[target]
        )
        for feature in features
    ]
})

In [98]:
df_compare = (df_spearman
    .merge(right = df_pearson, how = "inner", on = ["features"])
    .merge(right = df_mi     , how = "inner", on = ["features"])
    .merge(right = df_dcor   , how = "inner", on = ["features"]))

In [99]:
display(df_compare)

,features,spearman,pearson,mi,dcor
0,fast_trend,-0.047758,-0.055189,0.048265,0.158249
1,med_trend,-0.039134,-0.046177,0.066928,0.185108
2,slow_trend,-0.024493,-0.034022,0.091837,0.183382


In [100]:
print(os.getcwd())

A:\GitHub\SignalTrader\notebook
